### Setup

In [ ]:
!pip install huggingface_hub

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.4 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [ ]:
from huggingface_hub import datasets

ImportError: cannot import name 'datasets' from 'huggingface_hub' (/usr/local/lib/python3.10/dist-packages/huggingface_hub/__init__.py)

In [ ]:
%pip install --upgrade openai --quiet

In [ ]:
import json
import numpy as np
import openai
from IPython.display import Image, display, Audio, Markdown
import base64


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv()

## Set the API key and model name
MODEL="gpt-4o"

openai_api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=openai_api_key)

### Image Understanding

In [ ]:
segmented_file_path='/content/drive/MyDrive/bowen_research/mridata/segmented/'

In [ ]:
complete_diag_list[0]

'You are a radiologist. Given that the ACL shown in these MRI images is good, give a brief two-to-three sentence report of the findings and features'

In [ ]:
# read files from segmented_file_path into a dataframe, with file name in the img column, and the number before .jpg after _ as the value for diag
import os
import pandas as pd
files = os.listdir(segmented_file_path)
df = pd.DataFrame(columns=['img', 'diag'])
for file in files:
    if file.endswith('.jpg'):
        diag = int(file.split('.')[0][-1])
        df = pd.concat([df, pd.DataFrame({'img': [file], 'diag': [diag]})], ignore_index=True)

In [ ]:
df.head()

,img,diag
0,863766-3_segmented_2.jpg,2
1,800663-1_segmented_0.jpg,0
2,769102-0_segmented_1.jpg,1
3,847907-2_segmented_0.jpg,0
4,622688-2_segmented_0.jpg,0


In [ ]:
df_diag=df.copy()

In [ ]:
# Open the image file and encode it as a base64 string
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
for index, row in df_diag.iterrows():
  print(row['img'])
  base64_image = encode_image(segmented_file_path+row['img'])
  print(row['img'])
  complete_diag=f'You are a radiologist. Given that the ACL shown in these MRI images is {row.diag}, give a brief two-to-three sentence report of the findings and features'
  response = client.chat.completions.create(
      model=MODEL,
      messages=[
          {"role": "system", "content": "You are a helpful assistant that responds in Markdown. Help me with this MRI image!"},
          {"role": "user", "content": [
              {"type": "text", "text": complete_diag},
              {"type": "image_url", "image_url": {
                  "url": f"data:image/png;base64,{base64_image}"}
              }
          ]}
      ],
      temperature=0.0,
  )
  new_row={'img':row['img'],'diag':row['diag'],'report':response.choices[0].message.content}
  print(response.choices[0].message.content)
  df_diag.loc[len(df_diag)]=new_row


863766-3_segmented_2.jpg
863766-3_segmented_2.jpg
The MRI images demonstrate a complete tear of the anterior cruciate ligament (ACL), characterized by discontinuity and abnormal signal intensity in the expected location of the ligament. There is also evidence of joint effusion, which is commonly associated with ACL injuries. Further evaluation of associated structures, such as the menisci and collateral ligaments, is recommended to assess for additional injuries.
800663-1_segmented_0.jpg
800663-1_segmented_0.jpg
The MRI images show a well-defined anterior cruciate ligament (ACL) with normal signal intensity and intact fibers, indicating no signs of tear or degeneration. Surrounding structures, including the menisci and collateral ligaments, appear unremarkable. There is no evidence of joint effusion or bone marrow edema.
769102-0_segmented_1.jpg
769102-0_segmented_1.jpg
The MRI images reveal a partial tear of the anterior cruciate ligament (ACL), characterized by increased signal inten

In [ ]:
input='/content/drive/MyDrive/bowen_research/mridata/caption_output.txt'

In [ ]:
with open(input, 'r') as f:
    lines = f.readlines()

In [ ]:
df.head()

,img,diag
0,863766-3_segmented_2.jpg,2
1,800663-1_segmented_0.jpg,0
2,769102-0_segmented_1.jpg,1
3,847907-2_segmented_0.jpg,0
4,622688-2_segmented_0.jpg,0


In [ ]:
# Process the data into a list of tuples
data = []
for i in range(0, len(lines), 3):  # Iterate in steps of 3 (image, duplicate, caption)
    img = lines[i].strip()
    caption = lines[i + 2].strip()
    if (img, caption) not in data:  # Deduplicate entries
        data.append((img, caption))

# Create the DataFrame
df = pd.DataFrame(data, columns=["img", "report"])

In [ ]:
diag_map={0:'good',1:'partially torn',2:'completely torn'}

In [ ]:
df.head()

,img,report
0,863766-3_segmented_2.jpg,The MRI images demonstrate a complete tear of ...
1,800663-1_segmented_0.jpg,The MRI images show a well-defined anterior cr...
2,769102-0_segmented_1.jpg,The MRI images reveal a partial tear of the an...
3,847907-2_segmented_0.jpg,The MRI images demonstrate an intact anterior ...
4,622688-2_segmented_0.jpg,The MRI images demonstrate an intact anterior ...


In [ ]:
# add a new column to df_diag named report
df_diag.head()

,img,diag
0,863766-3_segmented_2.jpg,2
1,800663-1_segmented_0.jpg,0
2,769102-0_segmented_1.jpg,1
3,847907-2_segmented_0.jpg,0
4,622688-2_segmented_0.jpg,0


In [ ]:
df1=pd.merge(df,df_diag,on='img')

In [ ]:
df1.shape

(529, 3)

In [ ]:
df_diag.shape

(530, 2)

In [ ]:
df.shape

(529, 2)

In [ ]:
df['diag']=df_diag.diag.apply(lambda x: diag_map[x])

In [ ]:
df.head()

,img,report,diag
0,863766-3_segmented_2.jpg,The MRI images demonstrate a complete tear of ...,completely torn
1,800663-1_segmented_0.jpg,The MRI images show a well-defined anterior cr...,good
2,769102-0_segmented_1.jpg,The MRI images reveal a partial tear of the an...,partially torn
3,847907-2_segmented_0.jpg,The MRI images demonstrate an intact anterior ...,good
4,622688-2_segmented_0.jpg,The MRI images demonstrate an intact anterior ...,good


In [ ]:
df=df[['img','diag','report']]

In [ ]:
df.head()

,img,diag,report
0,863766-3_segmented_2.jpg,completely torn,The MRI images demonstrate a complete tear of ...
1,800663-1_segmented_0.jpg,good,The MRI images show a well-defined anterior cr...
2,769102-0_segmented_1.jpg,partially torn,The MRI images reveal a partial tear of the an...
3,847907-2_segmented_0.jpg,good,The MRI images demonstrate an intact anterior ...
4,622688-2_segmented_0.jpg,good,The MRI images demonstrate an intact anterior ...


In [ ]:
jpg_path='/content/drive/MyDrive/bowen_research/mridata/segmented/'

In [ ]:
df['image_name']=df.img.apply(lambda x: jpg_path+x)

In [ ]:
df.head()

,img,diag,report,image_name
0,863766-3_segmented_2.jpg,completely torn,The MRI images demonstrate a complete tear of ...,/content/drive/MyDrive/bowen_research/mridata/...
1,800663-1_segmented_0.jpg,good,The MRI images show a well-defined anterior cr...,/content/drive/MyDrive/bowen_research/mridata/...
2,769102-0_segmented_1.jpg,partially torn,The MRI images reveal a partial tear of the an...,/content/drive/MyDrive/bowen_research/mridata/...
3,847907-2_segmented_0.jpg,good,The MRI images demonstrate an intact anterior ...,/content/drive/MyDrive/bowen_research/mridata/...
4,622688-2_segmented_0.jpg,good,The MRI images demonstrate an intact anterior ...,/content/drive/MyDrive/bowen_research/mridata/...


In [ ]:
df1=df[['image_name','diag','report']]

In [ ]:
# do stratified split based on the value of diag for df_vol1_output
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df1, test_size=0.3, stratify=df1['diag'])

In [ ]:
df_train.diag.value_counts()

,count
diag,
good,275
partially torn,72
completely torn,23


In [ ]:
df_train.to_csv('/content/drive/MyDrive/bowen_research/mridata/train_data.csv',index=False)

In [ ]:
df_test.to_csv('/content/drive/MyDrive/bowen_research/mridata/test_data.csv',index=False)

In [ ]:
import pandas as pd

In [ ]:
df_train=pd.read_csv('/content/drive/MyDrive/bowen_research/mridata/train_data.csv')

In [ ]:
df_test=pd.read_csv('/content/drive/MyDrive/bowen_research/mridata/test_data.csv')

In [ ]:
df_train.head()

,image_name,diag,report
0,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images demonstrate an intact anterior ...
1,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images demonstrate an intact anterior ...
2,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images demonstrate an intact anterior ...
3,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images show a normal anterior cruciate...
4,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images show a normal anterior cruciate...


In [ ]:
df_test.drop(columns=['diag'],inplace=True)

In [ ]:
!pwd

/content


In [ ]:
train_captions.head()

,image_name,caption
0,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images demonstrate an intact anterior ...
1,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images reveal a partial tear of the an...
2,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images demonstrate a complete tear of ...
3,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images demonstrate a complete tear of ...
4,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images show a normal anterior cruciate...


In [ ]:
test_captions=pd.read_csv(image_dir+'test_captions_100.csv')

In [ ]:
test_captions.head()

,image_name,caption
0,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images demonstrate an intact anterior ...
1,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images demonstrate an intact anterior ...
2,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images reveal a partial tear of the an...
3,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images show a well-defined anterior cr...
4,/content/drive/MyDrive/bowen_research/mridata/...,The MRI images show an intact anterior cruciat...


In [ ]:
for df in [df_train,df_test]:
  df.rename({'report':'answer'},axis=1,inplace=True)
  df['question']='Assume you are an experienced radiologist, can you diagnose, based on this MRI image, if the ACL is good, partially torn, or completely torn? Please also give detailed explanation.'

In [ ]:
df_train.head()

,image_name,diag,answer,question
0,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images demonstrate an intact anterior ...,"Assume you are an experienced radiologist, can..."
1,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images demonstrate an intact anterior ...,"Assume you are an experienced radiologist, can..."
2,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images demonstrate an intact anterior ...,"Assume you are an experienced radiologist, can..."
3,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images show a normal anterior cruciate...,"Assume you are an experienced radiologist, can..."
4,/content/drive/MyDrive/bowen_research/mridata/...,good,The MRI images show a normal anterior cruciate...,"Assume you are an experienced radiologist, can..."


In [ ]:
df_train.drop(columns=['diag'],inplace=True)

In [ ]:
df_test.drop(columns=['diag'],inplace=True)

In [ ]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict, Features, Value, Image

# Function to load images
def load_image(image_path):
    with open(image_path, 'rb') as image_file:
        return image_file.read()

# Create a Huggingface dataset from the DataFrame
def create_dataset(df):
    # # Load the CSV file with captions
    # df = pd.read_csv(csv_file)
    # Filter out rows with empty captions (if any)
    df = df.dropna(subset=['answer'])
    # Load images and update the 'image' column
    df['image'] = df['image_name'].map(lambda image_name: load_image(image_name))
    # Remove the 'image_name' column as it's no longer needed
    df = df.drop(columns=['image_name'])
    # Reset the index to avoid '__index_level_0__' column
    df = df.reset_index(drop=True)
    # Define the features of the dataset
    features = Features({
        'image': Image(),
        'answer': Value('string'),
        'question': Value('string')
    })
    # Create a Dataset object from the DataFrame
    dataset = Dataset.from_pandas(df, features=features)
    return dataset

# Create train and test datasets
train_dataset = create_dataset(df_train)
test_dataset = create_dataset(df_test)

# Create DatasetDict
dataset_dict_train = DatasetDict({
    'train': train_dataset
})

dataset_dict_test = DatasetDict({
    'test': test_dataset
})

# Push to Huggingface (you need to be logged in)
dataset_dict_train.push_to_hub("Maymay07/mri_question_answer_train_full")
dataset_dict_test.push_to_hub("Maymay07/mri_question_answer_test_full")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/370 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/159 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

CommitInfo(commit_url='https://huggingface.co/datasets/Maymay07/mri_question_answer_test_full/commit/154e1a50e881bb4f8ca9ead7f6dde6e1c0ab7376', commit_message='Upload dataset', commit_description='', oid='154e1a50e881bb4f8ca9ead7f6dde6e1c0ab7376', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Maymay07/mri_question_answer_test_full', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Maymay07/mri_question_answer_test_full'), pr_revision=None, pr_num=None)

In [ ]:
!pip install huggingface

In [ ]:
!pip install datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 25.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.
